# PMM Dynamic Screener — MEXC Public REST

This notebook screens **MEXC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

This notebook is **public-data only**. It uses unauthenticated MEXC spot market endpoints and does not require or send any account credentials.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [1]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import MEXCPublicScreener, default_mexc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.mexc_public import MEXC_BASE_URL


PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "*"
INTERVAL = "1m"
UNIVERSE_TOP_K = 100
FINAL_TOP_N = 30
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "mexc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_mexc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 1000000.0
cfg.max_spread_bps = 50.0
cfg.min_top_of_book_quote = 250.0
cfg.min_depth_10bps_quote = 1000.0
cfg.max_last_trade_age_sec = 1800.0
cfg.min_recent_trade_count = 100
cfg.min_candle_count = 240
cfg.min_candle_coverage_ratio = 0.97
cfg.max_zero_volume_fraction = 0.2
cfg.min_natr_bps = 10.0
cfg.max_natr_bps = 250.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


Screener config


,value
connector,mexc
quote_asset,*
interval,1m
universe_top_k,100
final_top_n,30
candle_limit,288
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.12
timeout_seconds,30.0


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [3]:
screener = MEXCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,mexc,*,1m,2413,100


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,ETH-USDT,ETHUSDT,4.834973e+08,0.046106,99.877663,0.01000,0.000100,1
1,BTC-USDT,BTCUSDT,8.284443e+08,0.001402,99.873496,0.01000,0.000001,1
2,USDC-USDT,USDCUSDT,2.388074e+07,0.999850,99.577109,0.00010,1.000000,1
3,GOLD(PAXG)-USDT,GOLD(PAXG)USDT,1.546512e+07,0.022220,99.467272,0.01000,0.000001,1
4,GOLD(XAUT)-USDT,GOLD(XAUT)USDT,5.386325e+07,0.222521,99.301195,0.10000,0.001000,1
5,USD1-USDT,USD1USDT,9.560159e+06,1.000350,99.257929,0.00010,0.000000,1
6,BNB-USDT,BNBUSDT,2.314509e+07,0.154406,99.234946,0.01000,0.001000,1
7,XRP-USDT,XRPUSDT,6.856743e+07,1.413927,99.118680,0.00010,0.100000,1
8,DOGE-USDT,DOGEUSDT,3.161097e+07,1.040096,98.946628,0.00001,1.000000,1
9,USDE-USDT,USDEUSDT,4.150448e+06,1.000250,98.786530,0.00010,0.000000,1


Shortlist for detailed enrichment: 100


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,ETH-USDT,ETHUSDT,4.834973e+08,0.046106,99.877663,0.01000,0.000100,1
1,BTC-USDT,BTCUSDT,8.284443e+08,0.001402,99.873496,0.01000,0.000001,1
2,USDC-USDT,USDCUSDT,2.388074e+07,0.999850,99.577109,0.00010,1.000000,1
3,GOLD(PAXG)-USDT,GOLD(PAXG)USDT,1.546512e+07,0.022220,99.467272,0.01000,0.000001,1
4,GOLD(XAUT)-USDT,GOLD(XAUT)USDT,5.386325e+07,0.222521,99.301195,0.10000,0.001000,1
5,USD1-USDT,USD1USDT,9.560159e+06,1.000350,99.257929,0.00010,0.000000,1
6,BNB-USDT,BNBUSDT,2.314509e+07,0.154406,99.234946,0.01000,0.001000,1
7,XRP-USDT,XRPUSDT,6.856743e+07,1.413927,99.118680,0.00010,0.100000,1
8,DOGE-USDT,DOGEUSDT,3.161097e+07,1.040096,98.946628,0.00001,1.000000,1
9,USDE-USDT,USDEUSDT,4.150448e+06,1.000250,98.786530,0.00010,0.000000,1


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [4]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


,enriched_rows,selected_rows,pass_rate
0,100,11,0.11


,trading_pair,screen_score,passed_filters,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,WLD-USDT,75.208706,True,4.334214e+06,3.078344,1.262509e+03,1.452405e+04,500,11.957346,288,1.0,0.003472,13.067693,0.071533,
1,XLM-USDT,67.632176,True,7.458928e+06,5.635390,2.285632e+03,1.446896e+04,500,11.257934,288,1.0,0.003472,11.830729,0.000000,
2,DOT-USDT,66.410158,True,5.942912e+06,7.334067,4.136623e+03,4.136623e+03,500,4.005268,288,1.0,0.000000,14.545894,0.031142,
3,PEPE-USDT,64.085056,True,9.875603e+06,5.654509,4.255117e+02,4.497284e+03,500,24.104647,288,1.0,0.013889,10.269020,0.003425,
4,PUMP-USDT,63.123755,True,2.728063e+06,5.247966,7.840261e+02,9.037849e+03,500,19.034189,288,1.0,0.000000,15.277548,0.011628,
5,FET-USDT,62.265920,True,3.612312e+06,3.851338,6.210154e+02,2.910057e+03,500,5.803531,288,1.0,0.000000,31.307601,0.002674,
6,JST-USDT,54.755353,True,1.357469e+06,1.669867,3.311364e+02,1.499475e+03,500,16.589831,288,1.0,0.006944,10.384628,0.078845,
7,RENDER-USDT,51.765408,True,1.096228e+06,5.377790,7.053976e+02,7.327346e+03,500,8.734355,288,1.0,0.000000,12.912593,0.040650,
8,CRV-USDT,49.752072,True,1.118541e+06,4.256225,3.561272e+02,1.317887e+03,500,47.511124,288,1.0,0.000000,11.150053,0.033557,
9,OP-USDT,47.048008,True,1.185640e+06,8.814456,4.611672e+03,4.611672e+03,500,25.804487,288,1.0,0.038194,13.497043,0.008547,


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: {len(passed)} | Rejected: {len(rejected)}


,trading_pair,screen_score,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,natr_bps_mean,efficiency_ratio
0,WLD-USDT,75.208706,4.334214e+06,3.078344,1262.509416,14524.045701,500,11.957346,13.067693,0.071533
1,XLM-USDT,67.632176,7.458928e+06,5.635390,2285.632000,14468.960032,500,11.257934,11.830729,0.000000
2,DOT-USDT,66.410158,5.942912e+06,7.334067,4136.623220,4136.623220,500,4.005268,14.545894,0.031142
3,PEPE-USDT,64.085056,9.875603e+06,5.654509,425.511722,4497.283789,500,24.104647,10.269020,0.003425
4,PUMP-USDT,63.123755,2.728063e+06,5.247966,784.026067,9037.848564,500,19.034189,15.277548,0.011628
5,FET-USDT,62.265920,3.612312e+06,3.851338,621.015416,2910.056840,500,5.803531,31.307601,0.002674
6,JST-USDT,54.755353,1.357469e+06,1.669867,331.136400,1499.474993,500,16.589831,10.384628,0.078845
7,RENDER-USDT,51.765408,1.096228e+06,5.377790,705.397550,7327.346490,500,8.734355,12.912593,0.040650
8,CRV-USDT,49.752072,1.118541e+06,4.256225,356.127192,1317.886776,500,47.511124,11.150053,0.033557
9,OP-USDT,47.048008,1.185640e+06,8.814456,4611.671845,4611.671845,500,25.804487,13.497043,0.008547


,count
rejection_reason,
natr_bps_mean<10,58
top_of_book_quote<250,51
quote_volume_24h<1e+06,34
depth_10bps<1000,33
zero_volume_fraction>0.20,3


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [6]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=MEXC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/candle_ingestor_manife...
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260325_235511/exchange_rules_patch.yaml


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
WLD-USDT
XLM-USDT
DOT-USDT
PEPE-USDT
PUMP-USDT
FET-USDT
JST-USDT
RENDER-USDT
CRV-USDT
OP-USDT
ETC-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  mexc:
    enabled: true
    base_url: https://api.mexc.com
    pairs:
    - WLD/USDT
    - XLM/USDT
    - DOT/USDT
    - PEPE/USDT
    - PUMP/USDT
    - FET/USDT
    - JST/USDT
    - RENDER/USDT
    - CRV/USDT
    - OP/USDT
    - ETC/USDT
    intervals:
    - 1m
    trades:
      enabled: true
      limit: 500
      update_recent_candles: false
      recent_window_minutes: 120


Exchange rules patch (estimates only)
--------------------------------------------------------------